### IMPORTS from 

In [0]:
from pyspark.sql.functions import *

customers_df = spark.table("retails_catalog.silverlayers.silver_customers")
orders_df = spark.table("retails_catalog.silverlayers.silver_orders")
products_df = spark.table("retails_catalog.silverlayers.silver_products")
payments_df = spark.table("retails_catalog.silverlayers.silver_payments")
order_items_df = spark.table("retails_catalog.silverlayers.silver_order_items")

### STEP 1: Create DIMENSION TABLES

### 1.Customer Dimension

In [0]:
dim_customers = customers_df.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
).dropDuplicates(["customer_id"])

dim_customers.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retails_catalog.silverlayers.dim_customers")

### 2. Product Dimension

In [0]:
dim_products = products_df.select(
    "product_id",
    "product_category_name"
).dropDuplicates(["product_id"])

dim_products.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retails_catalog.silverlayers.dim_products")

###  STEP 2: Prepare FACT TABLE

### 1: Aggregate payments

In [0]:
from pyspark.sql.functions import sum

payments_agg = payments_df.groupBy("order_id").agg(
    sum("payment_value").alias("total_payment")
)

### 2: Join correctly

In [0]:
gold_fact_df = orders_df.alias("o") \
    .join(order_items_df.alias("oi"), "order_id") \
    .join(payments_agg.alias("pay"), "order_id") \
    .join(products_df.alias("p"), "product_id") \
    .join(customers_df.alias("c"), "customer_id")

### 3: Select only required columns

In [0]:
gold_fact_df = gold_fact_df.select(
    "order_id",
    "order_item_id",
    "customer_id",
    "product_id",
    "order_purchase_timestamp",
    "order_status",
    "price",
    "freight_value",
    "total_payment",
    "product_category_name",
    "customer_city",
    "customer_state"
)

### 4: Ensure no duplicates

In [0]:
gold_fact_df = gold_fact_df.dropDuplicates(
    ["order_id", "order_item_id"]
)

###  STEP 3: MERGE (INCREMENTAL LOAD)

In [0]:
from delta.tables import DeltaTable

table_name = "retails_catalog.silverlayers.fact_orders"

# Check if table exists
table_exists = spark.catalog.tableExists(table_name)

if table_exists:
    delta_table = DeltaTable.forName(spark, table_name)

    delta_table.alias("t").merge(
        gold_fact_df.alias("s"),
        "t.order_id = s.order_id AND t.order_item_id = s.order_item_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    gold_fact_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

###  STEP 4: KPI TABLES

### 1. Total Revenue

In [0]:
total_revenue_df = gold_fact_df.groupBy().sum("total_payment") \
    .withColumnRenamed("sum(total_payment)", "total_revenue")

total_revenue_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retails_catalog.silverlayers.kpi_total_revenue")

### 2. Top Customers

In [0]:
top_customers_df = gold_fact_df.groupBy("customer_id") \
    .sum("total_payment") \
    .withColumnRenamed("sum(total_payment)", "total_spent")

top_customers_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retails_catalog.silverlayers.kpi_top_customers")

### 3. Best Selling Products

In [0]:
best_products_df = gold_fact_df.groupBy("product_id", "product_category_name") \
    .sum("price") \
    .withColumnRenamed("sum(price)", "total_sales")

best_products_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retails_catalog.silverlayers.kpi_best_products")

### 4. Sales Trends (Monthly)

In [0]:
sales_trend_df = gold_fact_df.groupBy(
    year("order_purchase_timestamp").alias("year"),
    month("order_purchase_timestamp").alias("month")
).sum("total_payment") \
 .withColumnRenamed("sum(total_payment)", "monthly_sales")

sales_trend_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retails_catalog.silverlayers.kpi_sales_trends")

### 5. Payment Analysis

In [0]:
payment_analysis_df = payments_df.groupBy("payment_type") \
    .sum("payment_value") \
    .withColumnRenamed("sum(payment_value)", "payment_total")

payment_analysis_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retails_catalog.silverlayers.kpi_payment_analysis")

### 6. Customer Insights

In [0]:
customer_insights_df = gold_fact_df.groupBy("customer_id") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("total_payment").alias("total_spent")
    )

customer_insights_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retails_catalog.silverlayers.kpi_customer_insights")

###  FINAL VALIDATION

In [0]:
gold_fact_df.selectExpr(
    "count(*) as total_rows",
    "count(distinct order_id, order_item_id) as unique_rows"
).show()

### BUILT-IN DASHBOARD

### Step 1: Load Gold tables

In [0]:
fact_df = spark.read.format("delta").load("/mnt/gold/fact_orders")

top_customers_df = spark.read.format("delta").load("/mnt/gold/kpi_top_customers")

sales_trend_df = spark.read.format("delta").load("/mnt/gold/kpi_sales_trends")

category_df = spark.read.format("delta").load("/mnt/gold/kpi_best_products")

payment_df = spark.read.format("delta").load("/mnt/gold/kpi_payment_analysis")

### Step 2: Create charts (UI)

In [0]:
from pyspark.sql.functions import sum

sales_trend_year = spark.table("retails_catalog.silverlayers.kpi_sales_trends") \
    .groupBy("year") \
    .agg(sum("monthly_sales").alias("yearly_sales")) \
    .orderBy("year")

display(sales_trend_year)

Databricks visualization. Run in Databricks to view.

In [0]:

from pyspark.sql.functions import sum

customer_sales = spark.table("retails_catalog.silverlayers.kpi_top_customers") \
    .orderBy("total_spent", ascending=False) \
    .limit(10)

display(customer_sales)

Databricks visualization. Run in Databricks to view.

In [0]:
spark.table("retails_catalog.silverlayers.fact_orders").selectExpr("sum(total_payment) as total_revenue").display()

In [0]:
from pyspark.sql.functions import col

best_products = spark.table("retails_catalog.silverlayers.kpi_best_products") \
    .orderBy(col("total_sales").desc()) \
    .limit(10)

display(best_products)

Databricks visualization. Run in Databricks to view.

In [0]:
df = spark.table("retails_catalog.silverlayers.kpi_payment_analysis")
display(df)

Databricks visualization. Run in Databricks to view.

In [0]:
orders_df.write.format("delta").mode("overwrite").saveAsTable('retails_catalog.gold_layers.gold_orders')
customers_df.write.format("delta").mode("overwrite").saveAsTable('retails_catalog.gold_layers.gold_customers')
order_items_df.write.format("delta").mode("overwrite").saveAsTable('retails_catalog.gold_layers.gold_order_items')
payments_df.write.format("delta").mode("overwrite").saveAsTable('retails_catalog.gold_layers.gold_payments')
products_df.write.format("delta").mode("overwrite").saveAsTable('retails_catalog.gold_layers.gold_products')